In [4]:
# Aufgabe 9 PCA von Hand auf Basis der Korrelationsmatrix S, nicht auf Kovarianzmatrix C eingestellt mit jetzt ungleichen Varianzen

daten = [[2, 8], [8, 9], [8, 13]] # Beide Datensätze angelegt, um zwischen ihnen switchen zu können
#daten = [[6, 11], [5, 9], [4, 10]] 

def mittelwerte(daten):
    spaltenmittel = []
    summe_1 = 0
    summe_2 = 0
    for kd_merkmale in daten:
        summe_1 += kd_merkmale[0]
        summe_2 += kd_merkmale[1]
    
    spaltenmittel.append(summe_1 / len(daten))
    spaltenmittel.append(summe_2 / len(daten))
    
    return spaltenmittel

def zentriere(daten):
    mittel = mittelwerte(daten)
    zentrierte_daten = []
    for kd_daten in daten:
        z_wert_1 = kd_daten[0] - mittel[0]
        z_wert_2 = kd_daten[1] - mittel[1]

        z_daten = []
        z_daten.append(z_wert_1)
        z_daten.append(z_wert_2)

        zentrierte_daten.append(z_daten)

    return zentrierte_daten

def standardisiere(daten):
    std_abw = []
    X = zentriere(daten)
    n = len(X)
    for i in range(len(X[0])):
        summe = 0
        stdabw = 0
        for j in range(n):
            wert = X[j][i]**2
            summe += wert
        stdabw = (summe / (n-1))**0.5
        std_abw.append(stdabw)

    Xs = []
    for i in range(n):
        s_line = []
        for j in range(len(X[0])):
            s_wert = X[i][j] /std_abw[j]
            s_line.append(s_wert)
        Xs.append(s_line)

    return Xs

def spalte(daten, j):
    s_list = []
    for zeile in daten:
        s_list.append(zeile[j])
    return s_list

def korrelationsmatrix(Xs):
    S = []
    n = len(Xs)
    for i in range(len(Xs[0])):
        S_line = []
        sp_i = spalte(Xs, i) # eine Ebene höher, damit sie nicht jedes mal in der Schleife unten neu ermittelt wird obwohl sie auf der Ebene konstant bleibt.
        for j in range(len(Xs[0])):
            sp_j = spalte(Xs, j)
            
            summe = 0
            for k in range(n):
                summe += sp_i[k] * sp_j[k]

            korrelation = summe / (n-1)
            S_line.append(korrelation)
        S.append(S_line)

    return S

def kovarianzmatrix(X):
    C = []
    n = len(X)
    for i in range(len(X[0])):
        C_line = []
        sp_i = spalte(X, i) # eine Ebene höher, damit sie nicht jedes mal in der Schleife unten neu ermittelt wird obwohl sie auf der Ebene konstant bleibt.
        for j in range(len(X[0])):
            sp_j = spalte(X, j)
            
            summe = 0
            for k in range(n):
                summe += sp_i[k] * sp_j[k]

            covar = summe / (n-1)
            C_line.append(covar)
        C.append(C_line)

    return C

def eigenwerte_2x2(S):

    spur = S[0][0] + S[1][1]
    det = S[0][0] * S[1][1] - S[0][1] * S[1][0]

    sroot = ((spur**2 - 4*det)**0.5)

    lam_high = spur/2 + sroot/2
    lam_low = spur/2 - sroot/2

    return [lam_high, lam_low]

def eigenvektor_2x2(C, lam):
    v1 = C[0][1] # „Länge frei wählbar, hier so gesetzt, dass keine Division nötig ist — das Normieren macht die Wahl später gegenstandslos."
    v2 = lam - C[0][0]
    v_betrag = (v1**2 + v2**2)**0.5
    if v_betrag == 0:
        v1 = lam - C[1][1]
        v2 = C[1][0]
        v_betrag = (v1**2 + v2**2)**0.5
        if v_betrag == 0:
            v1 = 1
            v2 = 0
            v_betrag = 1 # Nicht mehr die Formel, da dieser Wert 1 sein muss bei gegebenem Vektor (1,0)

    return [v1 / v_betrag, v2 / v_betrag]

def scores(X, pcs):
    kd_scores_total = []
    for zeile in X:
        kd_scores = []
        for i in range(len(pcs)):
            kd_pcs = pcs[i]
            kd_score_pcs = 0
            for j in range(len(zeile)):
                kd_score_pcs += zeile[j] * kd_pcs[j]            
            kd_scores.append(kd_score_pcs)
        kd_scores_total.append(kd_scores)

    return kd_scores_total

def erklaerte_varianz(lams):
    gesamtvarianz = 0
    pc_beitraege = []
    for lam in lams:
        gesamtvarianz += lam
    for lam in lams:
        beitrag_n = lam / gesamtvarianz * 100
        pc_beitraege.append(beitrag_n)

    return pc_beitraege

def pca(daten):
    X = zentriere(daten)
    C = kovarianzmatrix(X)
    Xs = standardisiere(daten)
    S = korrelationsmatrix(Xs)
    lams = eigenwerte_2x2(S)
    lam_high = lams[0]
    lam_low = lams[1]
    eigvek_high = eigenvektor_2x2(S, lam_high)
    eigvek_low = eigenvektor_2x2(S, lam_low)
    pcs = [eigvek_high, eigvek_low]
    kunden_scores = scores(X, pcs)
    anteile = erklaerte_varianz(lams)

    pca_bestandteile = {'Kovarianzmatrix': C, 'Korrelationsmatrix': S, 'Eigenwerte': lams, 'PCs': pcs, 'Scores': kunden_scores, 'Anteile': anteile}

    return pca_bestandteile

print(pca(daten))



{'Kovarianzmatrix': [[12.0, 6.0], [6.0, 7.0]], 'Korrelationsmatrix': [[1.0000000000000002, 0.6546536707079772], [0.6546536707079772, 1.0]], 'Eigenwerte': [1.6546536707079769, 0.345346329292023], 'PCs': [[0.7071067811865479, 0.7071067811865472], [0.7071067811865476, -0.7071067811865476]], 'Scores': [[-4.242640687119286, -1.4142135623730951], [0.7071067811865486, 2.121320343559643], [3.5355339059327378, -0.7071067811865477]], 'Anteile': [82.73268353539885, 17.26731646460115]}
